<a href="https://colab.research.google.com/github/navya-goel28/deeplearning-using-pytorch/blob/main/qna_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [3]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
# vocab
vocab = {'<UNK>':0}

In [5]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [6]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
len(vocab)

324

In [8]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [9]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

In [11]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [12]:
dataset = QADataset(df, vocab)

In [13]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [14]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([162])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([100])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([316])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([185])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([106])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([244])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([36])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([121])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([268])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[10, 75, 76]]) tensor([77])
tensor([[  1,   2,   3, 163, 164, 165,  83,  84]]) tensor([166])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([54])
tensor([[10, 96,  3, 97]]) tensor([98])
tensor([[  1,   2,   3

In [15]:
import torch.nn as nn

In [16]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [17]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [18]:
learning_rate = 0.001
epochs = 20

In [19]:
model = SimpleRNN(len(vocab))

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [21]:
# training loop

for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()
    # forward pass
    output = model(question)
    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])
    # gradients
    loss.backward()
    # update
    optimizer.step()
    total_loss = total_loss + loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 520.149557
Epoch: 2, Loss: 452.457851
Epoch: 3, Loss: 373.058830
Epoch: 4, Loss: 311.783450
Epoch: 5, Loss: 259.672848
Epoch: 6, Loss: 210.885759
Epoch: 7, Loss: 167.109121
Epoch: 8, Loss: 129.568871
Epoch: 9, Loss: 98.462192
Epoch: 10, Loss: 75.002063
Epoch: 11, Loss: 57.946175
Epoch: 12, Loss: 45.298386
Epoch: 13, Loss: 36.270538
Epoch: 14, Loss: 29.633072
Epoch: 15, Loss: 24.762830
Epoch: 16, Loss: 20.858825
Epoch: 17, Loss: 17.790646
Epoch: 18, Loss: 15.224452
Epoch: 19, Loss: 13.110674
Epoch: 20, Loss: 11.409647


In [22]:
def predict(model, question, threshold=0.5):
  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)
  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  # send to model
  output = model(question_tensor)
  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)
  # find index of max prob
  value, index = torch.max(probs, dim=1)
  if value < threshold:
    print("I don't know")
  print(list(vocab.keys())[index])

In [28]:
predict(model, "chemical symbol for iron")

fe


In [31]:
list(vocab.keys())[25]

'point'